# Load Type Prediction – Machine Learning

This notebook predicts `Load_Type` (`Light_Load`, `Medium_Load`, `Maximum_Load`) from the provided power-system dataset. The **last month of data is kept as the final test set**, as required in the problem statement. fileciteturn0file0L2-L8

The workflow covers data cleaning, EDA, date/time feature engineering, model training, and classification evaluation using accuracy, precision, recall, and F1-score. The validation requirement specifies the last month as the test set. fileciteturn0file0L20-L24

## 1. Load the dataset

In [ ]:
DATA_FILE = 'load_data(1).csv'
df = pd.read_csv(DATA_FILE)
print('Shape:', df.shape)
display(df.head())
display(df.info())


The supplied data contains date/time, energy-consumption, reactive/power-factor, CO2, NSM, and `Load_Type` fields. The problem statement identifies `Load_Type` as the target with three categories. fileciteturn0file0L9-L19

In [ ]:
# Basic checks
print('Missing values:')
display(df.isna().sum())
print('\nTarget distribution:')
display(df['Load_Type'].value_counts())

print('\nDuplicate rows:', df.duplicated().sum())


## 2. Clean and prepare date/time data

The CSV contains a duplicate timestamp. We keep the first occurrence after sorting so that the time series has one record per timestamp. Missing numeric values are handled inside the ML pipeline using training-set medians.

In [ ]:
df['Date_Time'] = pd.to_datetime(df['Date_Time'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['Date_Time', 'Load_Type']).copy()
df = df.sort_values('Date_Time').drop_duplicates(subset='Date_Time', keep='first').reset_index(drop=True)

print('Cleaned shape:', df.shape)
print('Date range:', df['Date_Time'].min(), 'to', df['Date_Time'].max())


## 3. Exploratory data analysis

In [ ]:
df['Load_Type'].value_counts().plot(kind='bar', figsize=(7,4))
plt.title('Load Type Distribution')
plt.xlabel('Load Type')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
display(df[numeric_cols].describe().T)


## 4. Feature engineering

In [ ]:
def make_features(data):
    x = data.copy()
    dt = x['Date_Time']

    x['year'] = dt.dt.year
    x['month'] = dt.dt.month
    x['day'] = dt.dt.day
    x['dayofweek'] = dt.dt.dayofweek
    x['hour'] = dt.dt.hour
    x['minute'] = dt.dt.minute
    x['dayofyear'] = dt.dt.dayofyear
    x['weekofyear'] = dt.dt.isocalendar().week.astype(int)
    x['is_weekend'] = (dt.dt.dayofweek >= 5).astype(int)

    # Cyclic representation of time-of-day
    time_hours = dt.dt.hour + dt.dt.minute / 60.0
    x['hour_sin'] = np.sin(2 * np.pi * time_hours / 24)
    x['hour_cos'] = np.cos(2 * np.pi * time_hours / 24)

    return x.drop(columns=['Date_Time', 'Load_Type'])


## 5. Time-based train/test split

The problem statement explicitly requires the **last month** to be used as the test set rather than a random train/test split. fileciteturn0file0L20-L24

In [ ]:
last_month = df['Date_Time'].dt.to_period('M').max()
train_df = df[df['Date_Time'].dt.to_period('M') < last_month].copy()
test_df  = df[df['Date_Time'].dt.to_period('M') == last_month].copy()

X_train = make_features(train_df)
y_train = train_df['Load_Type']
X_test = make_features(test_df)
y_test = test_df['Load_Type']

print('Test month:', last_month)
print('Training rows:', len(train_df))
print('Test rows:', len(test_df))
print('Features:', list(X_train.columns))


## 6. Train the Random Forest classifier

Median imputation is fitted only on the training data through the pipeline, which avoids using test-set statistics during training. `class_weight='balanced'` helps account for the different class frequencies.

In [ ]:
model = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', RandomForestClassifier(
        n_estimators=250,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced'
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Model training completed.')


## 7. Evaluate the model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

metrics = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (weighted)', 'Recall (weighted)', 'F1-score (weighted)'],
    'Score': [accuracy, precision, recall, f1]
})
display(metrics)

print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
fig, ax = plt.subplots(figsize=(7,6))
disp.plot(ax=ax, xticks_rotation=30)
plt.title('Confusion Matrix – Last Month Test Set')
plt.tight_layout()
plt.show()


## 8. Feature importance

In [ ]:
rf = model.named_steps['classifier']
importance = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
display(importance.to_frame('importance').head(15))

importance.head(15).sort_values().plot(kind='barh', figsize=(8,6))
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 9. Save the trained model (optional)
The saved pipeline includes both missing-value handling and the Random Forest model.

In [ ]:
import joblib
joblib.dump(model, 'load_type_random_forest.joblib')
print('Saved: load_type_random_forest.joblib')
